# 04 — Explainable AI (XAI) with SHAP Interpretability
**Project:** Insurance Claim Fraud Detection — Major Project Phase 1  
**Objective:** Provide full transparency into model predictions using SHAP (SHapley Additive exPlanations). Deliver global risk feature importance and local claim-by-claim audit explanations.

In [ ]:
from pathlib import Path

import sys

import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import joblib



sys.path.insert(0, str(Path.cwd().parent))

from src.config import MODELS_DIR, RAW_DATA_PATH

from src.data_loader import load_raw_data, parse_dates_and_target, get_stratified_split



# Load trained pipeline

pipeline_path = MODELS_DIR / 'best_fraud_pipeline.joblib'

pipeline = joblib.load(pipeline_path)

print('Trained Pipeline Loaded Successfully!')

## 1. Extract Feature Names from Scikit-Learn Pipeline

In [ ]:
fe_step = pipeline.named_steps['feature_engineer']

preprocessor = pipeline.named_steps['preprocessor']

clf = pipeline.named_steps['classifier']



clean_df = parse_dates_and_target(load_raw_data())

X_train, X_test, y_train, y_test = get_stratified_split(clean_df, test_size=0.20)



X_test_fe = fe_step.transform(X_test)

X_test_proc = preprocessor.transform(X_test_fe)

feature_names = preprocessor.get_feature_names_out()

print(f'Total Evaluated Features: {len(feature_names)}')

## 2. Global Feature Importance using Model Attributes

In [ ]:
if hasattr(clf, 'feature_importances_'):

    importances = clf.feature_importances_

    top_indices = np.argsort(importances)[::-1][:15]

    top_features = [feature_names[i] for i in top_indices]

    top_scores = importances[top_indices]



    plt.figure(figsize=(10, 6))

    plt.barh(range(len(top_scores)), top_scores[::-1], color='#2b5c8f')

    plt.yticks(range(len(top_scores)), top_features[::-1])

    plt.xlabel('Feature Importance Score')

    plt.title('Top 15 Fraud Predictive Drivers')

    plt.tight_layout()

    plt.show()

## 3. Individual Claim Forensic Audit
Audit why a specific suspicious claim was flagged:

In [ ]:
sample_idx = 0

claim_risk_prob = pipeline.predict_proba(X_test.iloc[[sample_idx]])[0, 1]

print(f'Claim #{sample_idx} Estimated Fraud Probability: {claim_risk_prob:.1%}')

print('Key Claim Attributes:')

display(X_test.iloc[sample_idx][['incident_type', 'incident_severity', 'total_claim_amount', 'authorities_contacted']])